# Day 4: Spark SQL and Complex Joins Practice Exercises

## Welcome!
These exercises help beginners practice Spark SQL and complex joins using the COVID-19 dataset.

## Before You Start
- Run `docker-compose up` in the `01_basic_spark` directory.
- Open Jupyter at `http://localhost:8888`.
- Place `covid-data.csv` in the `covid-dataset/` directory.
- Please refer [week-2 materials](https://github.com/LD-LINC/Week2---SQL-And-Data-Modeling) and [exercises](https://github.com/LD-LINC/Week2---SQL-And-Data-Modeling/tree/main/05_weekly_project) for better understanding of SQL.
- Download the dataset if needed: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

## Exercises


In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
f = F = sf 

---

### Exercise 1: Create a Temporary View
#### What to Do
- Load `covid-data.csv` into a DataFrame and create a temporary view named `covid_view`.
#### Steps
- Start a Spark session.
- Read the CSV with `header=True` and `inferSchema=True`.
- Create a temporary view called `covid_view`.
- Show the first 5 rows to verify.



In [2]:
spark = SparkSession.builder.appName("Spark-SQL-Exercises").getOrCreate()

In [33]:
df = df_org = spark.read.csv("data/owid-covid-data.csv", header=True, inferSchema=True)
df = df_org.filter(f.col("continent").isNotNull())  # remove rows where locations are continents

In [37]:
#df.printSchema()

In [ ]:
#df.show()

In [25]:
df_org.createOrReplaceTempView("covid_view_org")

In [26]:
df.createOrReplaceTempView("covid_view")

In [27]:
spark.sql("""
SELECT iso_code, continent, location, date FROM covid_view LIMIT 5
""").show()

+--------+---------+-----------+----------+
|iso_code|continent|   location|      date|
+--------+---------+-----------+----------+
|     AFG|     Asia|Afghanistan|2020-01-05|
|     AFG|     Asia|Afghanistan|2020-01-06|
|     AFG|     Asia|Afghanistan|2020-01-07|
|     AFG|     Asia|Afghanistan|2020-01-08|
|     AFG|     Asia|Afghanistan|2020-01-09|
+--------+---------+-----------+----------+



### Data contains also locations for continents

these have continent=NULL and should be filtered before-hand

In [30]:
spark.sql("""
SELECT DISTINCT continent, location
FROM  covid_view_org 
WHERE continent IS NULL
""").show()

+---------+--------------------+
|continent|            location|
+---------+--------------------+
|     NULL|              Africa|
|     NULL|                Asia|
|     NULL|              Europe|
|     NULL| European Union (27)|
|     NULL|High-income count...|
|     NULL|Low-income countries|
|     NULL|Lower-middle-inco...|
|     NULL|       North America|
|     NULL|             Oceania|
|     NULL|       South America|
|     NULL|Upper-middle-inco...|
|     NULL|               World|
+---------+--------------------+



In [31]:
df = df_org.filter(f.col("continent").isNotNull())

In [32]:
df.createOrReplaceTempView("covid_view")

---

### Exercise 2: Basic SQL Query
#### What to Do
- Use Spark SQL to select `location`, `date`, and `new_cases` where `new_cases` > 1000.
#### Steps
- Use `spark.sql()` to query `covid_view`.
- Filter rows where `new_cases > 1000`.
- Show the first 10 rows.


In [112]:
spark.sql("""
SELECT location, date, new_cases
FROM covid_view
WHERE new_cases > 100
""").show(10)

+-----------+----------+---------+
|   location|      date|new_cases|
+-----------+----------+---------+
|Afghanistan|2020-04-05|      183|
|Afghanistan|2020-04-12|      247|
|Afghanistan|2020-04-19|      387|
|Afghanistan|2020-04-26|      422|
|Afghanistan|2020-05-03|      841|
|Afghanistan|2020-05-10|     1392|
|Afghanistan|2020-05-17|     2490|
|Afghanistan|2020-05-24|     3813|
|Afghanistan|2020-05-31|     4577|
|Afghanistan|2020-06-07|     5108|
+-----------+----------+---------+
only showing top 10 rows



---

### Exercise 3: Group By with SQL
#### What to Do
- Calculate the total `new_cases` per `continent` using Spark SQL.
#### Steps
- Write a SQL query to group by `continent` and sum `new_cases`.
- Show the results.



In [113]:
spark.sql("""
SELECT continent, SUM(new_cases) AS total_new_cases
FROM covid_view AS c
GROUP BY continent
""").show(10)

+-------------+---------------+
|    continent|total_new_cases|
+-------------+---------------+
|       Europe|      252916868|
|       Africa|       13146831|
|North America|      124492698|
|South America|       68811012|
|      Oceania|       15003468|
|         Asia|      301564180|
+-------------+---------------+



---

### Exercise 4: Filter with WHERE Clause
#### What to Do
- Find records where `reproduction_rate` > 1.2 and `continent` is not null, showing `location`, `date`, `reproduction_rate`.
#### Steps
- Write a SQL query with a `WHERE` clause for `reproduction_rate > 1.2` and `continent IS NOT NULL`.
- Select the specified columns and show 10 rows.


In [114]:
spark.sql("""
SELECT location, date, reproduction_rate
FROM covid_view AS c
WHERE reproduction_rate > 1.2 
    AND continent IS NOT NULL
""").show(10)

+-----------+----------+-----------------+
|   location|      date|reproduction_rate|
+-----------+----------+-----------------+
|Afghanistan|2020-03-29|             1.51|
|Afghanistan|2020-03-30|             1.51|
|Afghanistan|2020-03-31|             1.52|
|Afghanistan|2020-04-01|             1.51|
|Afghanistan|2020-04-02|             1.51|
|Afghanistan|2020-04-03|              1.5|
|Afghanistan|2020-04-04|             1.49|
|Afghanistan|2020-04-05|             1.49|
|Afghanistan|2020-04-06|             1.49|
|Afghanistan|2020-04-07|             1.49|
+-----------+----------+-----------------+
only showing top 10 rows



---

### Exercise 5: Average KPI with SQL
#### What to Do
- Calculate the average `total_cases_per_million` per `continent` using Spark SQL, sorted descending.
#### Steps
- Use `AVG()` and `GROUP BY continent`.
- Sort by average cases in descending order.
- Show the results.


In [115]:
spark.sql("""
SELECT continent, ROUND(AVG(total_cases_per_million),2) AS avg_cases
FROM covid_view AS c
GROUP BY continent
""").show(10)

+-------------+---------+
|    continent|avg_cases|
+-------------+---------+
|       Europe|224006.97|
|       Africa| 26604.43|
|North America|132425.64|
|South America|111028.53|
|      Oceania|113796.87|
|         Asia| 80234.35|
+-------------+---------+




---

### Exercise 6: HAVING Clause
#### What to Do
- Find continents with average `total_deaths_per_million` > 500, showing only `continent` and the average.
#### Steps
- Write a SQL query with `GROUP BY continent` and `HAVING` clause.
- Show the results.


In [116]:
spark.sql("""
SELECT continent, ROUND(AVG(total_deaths_per_million), 2) AS avg_deaths
FROM covid_view AS c
GROUP BY continent
HAVING AVG(total_deaths_per_million) > 500
""").show(10)

+-------------+----------+
|    continent|avg_deaths|
+-------------+----------+
|       Europe|   1758.85|
|North America|    967.81|
|South America|   1687.42|
+-------------+----------+



---

### Exercise 7: Inner Join with SQL  
#### What to Do  
- Create two views from `covid_view`: one for 2020 data and one for 2021 data. Perform an inner join on `location` where `total_cases > 100000` in both years.  

#### Steps  
- Create view `covid_2020` with `location` and `MAX(total_cases)` for 2020 where `total_cases > 100000`.  
- Create view `covid_2021` with `location` and `MAX(total_cases)` for 2021 where `total_cases > 100000`.  
- Perform inner join between `covid_2020` and `covid_2021` on `location`.  
- Select `location`, `cases_2020`, and `cases_2021`.  
- Show top 10 rows.  



In [117]:
spark.sql("""
SELECT location, MAX(total_cases) AS max_total_cases
FROM covid_view AS c
WHERE YEAR(date) = 2020
GROUP BY location
HAVING max_total_cases > 100000
ORDER BY location
--ORDER BY max_total_cases DESC
""").createOrReplaceTempView("covid_2020")

spark.sql("SELECT * FROM covid_2020").show(5)

+----------+---------------+
|  location|max_total_cases|
+----------+---------------+
| Argentina|        1629908|
|   Armenia|         157834|
|   Austria|         344732|
|Azerbaijan|         211764|
|Bangladesh|         509148|
+----------+---------------+
only showing top 5 rows



In [118]:
spark.sql("""
SELECT location, MAX(total_cases) AS max_total_cases
FROM covid_view AS c
WHERE YEAR(date) = 2021
GROUP BY location
HAVING max_total_cases > 100000
ORDER BY location
--ORDER BY max_total_cases DESC
""").createOrReplaceTempView("covid_2021")

spark.sql("SELECT * FROM covid_2021").show(5)

+-----------+---------------+
|   location|max_total_cases|
+-----------+---------------+
|Afghanistan|         157902|
|    Albania|         207221|
|    Algeria|         216376|
|  Argentina|        5559916|
|    Armenia|         344481|
+-----------+---------------+
only showing top 5 rows



In [119]:
spark.sql("""
SELECT 
    c20.location AS location,
    c20.max_total_cases as cases_2020,
    c21.max_total_cases as cases_2021
    
FROM covid_2020 AS c20
INNER JOIN covid_2021 as c21
    ON c20.location = c21.location
--ORDER BY location
""").show(10)

+--------------------+----------+----------+
|            location|cases_2020|cases_2021|
+--------------------+----------+----------+
|           Argentina|   1629908|   5559916|
|          Azerbaijan|    211764|    614119|
|             Armenia|    157834|    344481|
|             Austria|    344732|   1252088|
|             Belgium|    638760|   2048110|
|             Belarus|    184922|    692601|
|             Bolivia|    153590|    575247|
|          Bangladesh|    509148|   1583253|
|              Brazil|   7448560|  22230737|
|Bosnia and Herzeg...|    109330|    287716|
+--------------------+----------+----------+
only showing top 10 rows




---

### Exercise 8: Left Join with SQL
#### What to Do
- Perform a left join between `covid_view` (all records) and a view of locations with `new_deaths` > 1000, showing `location`, `date`, and `new_deaths`.
#### Steps
- Create a view for records with `new_deaths > 1000`.
- Perform a left join with `covid_view` on `location` and `date`.
- Show 10 rows.


In [92]:
df = spark.read.csv("data/owid-covid-data.csv", header=True, inferSchema=True)
df = df.filter(f.col("continent").isNotNull())
df.createOrReplaceTempView("covid_view")

In [123]:
spark.sql("""
SELECT location, date, new_deaths
FROM covid_view AS c
WHERE new_deaths > 1000
""").createOrReplaceTempView("covid_new_deaths_gt_1000")

In [124]:
spark.sql("""
SELECT DISTINCT c.location, c.date, c1.new_deaths as new_deaths

FROM covid_view AS c
RIGHT JOIN covid_new_deaths_gt_1000 as c1
    ON c.location = c1.location
    AND c.date = c1.date
    
ORDER BY new_deaths DESC
""").show(10)

+-------------+----------+----------+
|     location|      date|new_deaths|
+-------------+----------+----------+
|        China|2023-02-05|     47687|
|        India|2021-05-23|     28982|
|        India|2021-05-16|     27922|
|        India|2021-05-09|     26820|
|        India|2021-05-30|     26706|
|        India|2021-06-13|     23625|
|United States|2021-01-17|     23312|
|        India|2021-05-02|     23231|
|United States|2021-01-24|     22495|
|United States|2021-01-31|     22249|
+-------------+----------+----------+
only showing top 10 rows



---

### Exercise 9: Right Join with SQL
#### What to Do
- Create a view for locations with `total_vaccinations` > 1000000. Perform a right join with `covid_view` on `location`, showing `location`, `total_vaccinations`, and `total_cases`.
#### Steps
- Create a view for high-vaccination locations.
- Perform a right join with `covid_view`.
- Show 10 rows.


In [130]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW  covid_high_vaccinations  AS
SELECT DISTINCT location, total_vaccinations
FROM covid_view
WHERE total_vaccinations > 1000000
""")

DataFrame[]

In [134]:
spark.sql("""
SELECT DISTINCT c.location, v.total_vaccinations, c.total_cases
FROM covid_view AS c
RIGHT JOIN covid_high_vaccinations AS v
    ON c.location = v.location
""").show()

+----------+------------------+-----------+
|  location|total_vaccinations|total_cases|
+----------+------------------+-----------+
| Argentina|          91307059|    9101319|
| Argentina|         116887049|   10099939|
| Argentina|         116887049|   10057795|
| Argentina|         116887049|    2647413|
| Australia|          23100911|   11061493|
| Australia|          23100911|     211425|
| Australia|          23100911|      37775|
| Australia|          41251388|    3454398|
|Azerbaijan|          11177159|     793695|
|Azerbaijan|          11177159|      29312|
|   Albania|           2634377|     155293|
| Argentina|          75053943|    9101319|
| Argentina|          85129838|   10032709|
| Argentina|         116108553|     524195|
| Argentina|         116732777|   10040329|
| Australia|          53866202|   10365801|
| Australia|          60430700|    9811888|
| Australia|          60430700|       7255|
| Australia|          68770769|   10365801|
|   Austria|          19021746| 


---

### Exercise 10: Full Outer Join  
#### What to Do  
- Perform a full outer join between views of 2020 and 2021 data on `location`, showing `location`, `total_cases` (2020), and `total_cases` (2021).  

#### Steps  
- Create view `covid_2020` with `location` and `MAX(total_cases)` for year 2020.  
- Create view `covid_2021` with `location` and `MAX(total_cases)` for year 2021.  
- Perform a `FULL OUTER JOIN` between `covid_2020` and `covid_2021` on `location`.  
- Use `COALESCE` to handle `location` values from either side.  
- Select `location`, `total_cases_2020`, and `total_cases_2021`.  
- Show top 10 rows.  



In [9]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW covid_2020 AS
SELECT location, MAX(total_cases) AS total_cases_2020
FROM covid_view
WHERE YEAR(date) = 2020
GROUP BY location
""")

DataFrame[]

In [10]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW covid_2021 AS
SELECT location, MAX(total_cases) AS total_cases_2021
FROM covid_view
WHERE YEAR(date) = 2021
GROUP BY location
""")

DataFrame[]

In [34]:
spark.sql("""
SELECT 
    COALESCE(c20.location, c21.location) AS location,
    --c20.location as location_20,
    --c21.location as location_21,
    total_cases_2020,
    total_cases_2021
FROM covid_2020 AS c20
FULL OUTER JOIN covid_2021 AS c21
    ON c20.location = c21.location
--WHERE c20.location IS NULL 
--    OR c21.location IS NULL
--    OR total_cases_2020 IS NULL
--    OR total_cases_2021 IS NULL
""").show(10)

+-------------------+----------------+----------------+
|           location|total_cases_2020|total_cases_2021|
+-------------------+----------------+----------------+
|        Afghanistan|           51848|          157902|
|            Albania|           55380|          207221|
|            Algeria|           97857|          216376|
|     American Samoa|               0|              11|
|            Andorra|            7806|           21730|
|             Angola|           17149|           71142|
|           Anguilla|              12|            1646|
|Antigua and Barbuda|             155|            4229|
|          Argentina|         1629908|         5559916|
|            Armenia|          157834|          344481|
+-------------------+----------------+----------------+
only showing top 10 rows



---

### Exercise 11: Self Join
#### What to Do
- Use a self join on `covid_view` to compare `total_cases` for each location between consecutive dates.
#### Steps
- Write a SQL query joining `covid_view` with itself on `location` and consecutive `date`.
- Calculate the difference in `total_cases`.
- Show 10 rows with `location`, `date` (first), `date` (second), and case difference.


spark.sql("""
SELECT 
FROM
WHERE
GROUP BY
HAVING
ORDER BY
""")

In [35]:
result = spark.sql("""
SELECT c1.location,
       c1.date AS date1,
       c2.date AS date2,
       (c2.total_cases - c1.total_cases) AS case_diff
FROM covid_view c1
JOIN covid_view c2
  ON c1.location = c2.location
 AND DATEDIFF(c2.date, c1.date) = 1
WHERE c1.total_cases IS NOT NULL
  AND c2.total_cases IS NOT NULL
""")

# Show top 10 rows where the difference is not zero
result.filter("case_diff!=0").show(10)


+-----------+----------+----------+---------+
|   location|     date1|     date2|case_diff|
+-----------+----------+----------+---------+
|Afghanistan|2020-02-29|2020-03-01|        1|
|Afghanistan|2020-03-14|2020-03-15|        6|
|Afghanistan|2020-03-21|2020-03-22|       17|
|Afghanistan|2020-03-28|2020-03-29|       67|
|Afghanistan|2020-04-04|2020-04-05|      183|
|Afghanistan|2020-04-11|2020-04-12|      247|
|Afghanistan|2020-04-18|2020-04-19|      387|
|Afghanistan|2020-04-25|2020-04-26|      422|
|Afghanistan|2020-05-02|2020-05-03|      841|
|Afghanistan|2020-05-09|2020-05-10|     1392|
+-----------+----------+----------+---------+
only showing top 10 rows



---

### Exercise 12: Window Function - Rank
#### What to Do
- Rank locations by `total_cases` within each `continent`, showing only rank 1.
#### Steps
- Use `RANK()` over a window partitioned by `continent`.
- Filter for rank = 1.
- Show `continent`, `location`, `total_cases`, and rank.


spark.sql("""
SELECT 
FROM
WHERE
GROUP BY
HAVING
ORDER BY
""")

In [37]:
spark.sql("""
SELECT continent, location, total_cases, rank
FROM (
    SELECT 
        continent,
        location,
        MAX(total_cases) AS total_cases,
        RANK() OVER (PARTITION BY continent ORDER BY MAX(total_cases) DESC) AS rank
    FROM covid_view
    GROUP BY continent, location
)
WHERE rank = 1
""").show()

+-------------+-------------+-----------+----+
|    continent|     location|total_cases|rank|
+-------------+-------------+-----------+----+
|       Africa| South Africa|    4072765|   1|
|         Asia|        China|   99373219|   1|
|       Europe|       France|   38997490|   1|
|North America|United States|  103436829|   1|
|      Oceania|    Australia|   11861161|   1|
|South America|       Brazil|   37511921|   1|
+-------------+-------------+-----------+----+





---

### Exercise 13: Window Function - Running Total
#### What to Do
- Calculate a running total of `new_cases` for each `location` over time using a window function.
#### Steps
- Use `SUM(new_cases)` over a window partitioned by `location` and ordered by `date`.
- Show `location`, `date`, `new_cases`, and running total.
- Show 10 rows.


spark.sql("""
SELECT 
FROM
WHERE
GROUP BY
HAVING
ORDER BY
""")

In [50]:
# Calculate running total of new_cases per location using window function
#   'PARTITION BY location' ensures each location is computed separately
#   'ORDER BY date' ensures cumulative sum follows chronological order
#   'ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW' accumulates from start to current row
result = spark.sql("""
SELECT location,
       date,
       new_cases,
       SUM(new_cases) OVER (
           PARTITION BY location
           ORDER BY date
           
           -- ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           
           ROWS BETWEEN 
               
               UNBOUNDED PRECEDING  -- means 'all preceding', to limit use e.g. '3 PRECEDING'
               
               AND
               
               CURRENT ROW          -- or use '1 FOLLOWING', 'UNBOUNDED FOLLOWING', ...
               
       ) AS running_total
FROM covid_view
WHERE new_cases IS NOT NULL
    AND new_cases > 0
    AND location LIKE 'Germ%'
    
ORDER BY location, date DESC
--ORDER BY running_total DESC
""")

# Display top 20 rows, full content without truncation
result.show(20, False)

+--------+----------+---------+-------------+
|location|date      |new_cases|running_total|
+--------+----------+---------+-------------+
|Germany |2023-07-02|1099     |38437756     |
|Germany |2023-06-25|1528     |38436657     |
|Germany |2023-06-18|2014     |38435129     |
|Germany |2023-06-11|2392     |38433115     |
|Germany |2023-06-04|2767     |38430723     |
|Germany |2023-05-28|4272     |38427956     |
|Germany |2023-05-21|4359     |38423684     |
|Germany |2023-05-14|6708     |38419325     |
|Germany |2023-05-07|7360     |38412617     |
|Germany |2023-04-30|9193     |38405257     |
|Germany |2023-04-23|11468    |38396064     |
|Germany |2023-04-16|13045    |38384596     |
|Germany |2023-04-09|14679    |38371551     |
|Germany |2023-04-02|22652    |38356872     |
|Germany |2023-03-26|32639    |38334220     |
|Germany |2023-03-19|40307    |38301581     |
|Germany |2023-03-12|43758    |38261274     |
|Germany |2023-03-05|86321    |38217516     |
|Germany |2023-02-26|113434   |381


---

### Exercise 14: Window Function - Moving Average
#### What to Do
- Calculate a 7-day moving average of `new_cases` for each `location`.
#### Steps
- Use `AVG(new_cases)` over a window partitioned by `location`, ordered by `date`, with a 7-day range.
- Show `location`, `date`, `new_cases`, and moving average.
- Show 10 rows.




In [69]:

# Actually calculate 2-week running avg, because data is recorded only once a week

# PARTITION BY location ensures calculation per location
# ORDER BY date ensures correct chronological order
# ROWS BETWEEN 13 PRECEDING AND CURRENT ROW takes current and 13 previous rows (14 days)
result = spark.sql("""
WITH data AS (

SELECT location,
       date,
       new_cases,
       AVG(new_cases) OVER (
           PARTITION BY location
           ORDER BY date
           ROWS BETWEEN 13 PRECEDING AND CURRENT ROW
       ) AS moving_avg
FROM covid_view
WHERE new_cases IS NOT NULL
    --AND new_cases > 0
    AND (
    location LIKE 'United S%'
    --OR location LIKE 'Germ%'
    )
    
ORDER BY location, date DESC
--ORDER BY moving_avg DESC

)
SELECT location,
       date,
       new_cases,
       ROUND(moving_avg, 2) as moving_avg
FROM data
WHERE moving_avg > 0
""")

# Display top 10 rows
result.show(30)


+-------------+----------+---------+----------+
|     location|      date|new_cases|moving_avg|
+-------------+----------+---------+----------+
|United States|2023-05-20|        0|  12173.21|
|United States|2023-05-19|        0|  12173.21|
|United States|2023-05-18|        0|  12173.21|
|United States|2023-05-17|        0|  12173.21|
|United States|2023-05-16|        0|  12173.21|
|United States|2023-05-15|        0|  12173.21|
|United States|2023-05-14|    93260|  12173.21|
|United States|2023-05-13|        0|  11689.21|
|United States|2023-05-12|        0|  11689.21|
|United States|2023-05-11|        0|  11689.21|
|United States|2023-05-10|        0|  11689.21|
|United States|2023-05-09|        0|  11689.21|
|United States|2023-05-08|        0|  11689.21|
|United States|2023-05-07|    77165|  11689.21|
|United States|2023-05-06|        0|  13210.79|
|United States|2023-05-05|        0|  13210.79|
|United States|2023-05-04|        0|  13210.79|
|United States|2023-05-03|        0|  13

---

### Exercise 15: Case Statement
#### What to Do
- Use a `CASE` statement to categorize `reproduction_rate` into 'Low' (<1), 'Medium' (1-1.5), and 'High' (>1.5).
#### Steps
- Write a SQL query with a `CASE` statement.
- Show `location`, `date`, `reproduction_rate`, and category.
- Show 10 rows.


In [70]:
result = spark.sql("""
SELECT location,
       date,
       reproduction_rate,
       CASE
           WHEN reproduction_rate < 1 THEN 'Low'
           WHEN reproduction_rate BETWEEN 1 AND 1.5 THEN 'Medium'
           WHEN reproduction_rate > 1.5 THEN 'High'
           ELSE 'Unknown'
       END AS rate_category
FROM covid_view
WHERE reproduction_rate IS NOT NULL
""")

# Show top 10 rows
result.show(10)

+-----------+----------+-----------------+-------------+
|   location|      date|reproduction_rate|rate_category|
+-----------+----------+-----------------+-------------+
|Afghanistan|2020-03-29|             1.51|         High|
|Afghanistan|2020-03-30|             1.51|         High|
|Afghanistan|2020-03-31|             1.52|         High|
|Afghanistan|2020-04-01|             1.51|         High|
|Afghanistan|2020-04-02|             1.51|         High|
|Afghanistan|2020-04-03|              1.5|       Medium|
|Afghanistan|2020-04-04|             1.49|       Medium|
|Afghanistan|2020-04-05|             1.49|       Medium|
|Afghanistan|2020-04-06|             1.49|       Medium|
|Afghanistan|2020-04-07|             1.49|       Medium|
+-----------+----------+-----------------+-------------+
only showing top 10 rows



---

### Exercise 16: Subquery for KPI
#### What to Do
- Find the date with the highest `new_cases` for each `location` using a subquery.
#### Steps
- Write a SQL query with a subquery to find max `new_cases` per `location`.
- Join with `covid_view` to get the corresponding `date`.
- Show `location`, `date`, and `new_cases`.


In [ ]:
spark.sql("""
SELECT 
FROM
WHERE
GROUP BY
HAVING
ORDER BY
""")

---

### Exercise 17: Compare SQL and DataFrame API
#### What to Do
- Calculate the maximum `new_deaths` per `continent` using both Spark SQL and DataFrame API.
#### Steps
- Write a SQL query with `MAX(new_deaths)` and `GROUP BY continent`.
- Use DataFrame API with `groupBy()` and `agg()`.
- Show both results.



In [ ]:
spark.sql("""
SELECT 
FROM
WHERE
GROUP BY
HAVING
ORDER BY
""")

---

### Exercise 18: Cross Join for Combinations
#### What to Do
- Create a view of distinct continents and perform a cross join with a view of years (2020, 2021) to list all continent-year combinations.
#### Steps
- Create a view for distinct `continent` values.
- Create a view for years `[2020, 2021]`.
- Perform a cross join and show all combinations.



In [ ]:
spark.sql("""
SELECT 
FROM
WHERE
GROUP BY
HAVING
ORDER BY
""")

---

### Exercise 19: KPI - Case Fatality Rate
#### What to Do
- Calculate the case fatality rate (`total_deaths` / `total_cases`) per `location` for the latest date.
#### Steps
- Use a subquery to find the latest date per `location`.
- Calculate the fatality rate, handling nulls and division by zero.
- Show `location`, `date`, and fatality rate.



In [ ]:
spark.sql("""
SELECT 
FROM
WHERE
GROUP BY
HAVING
ORDER BY
""")

---

### Exercise 20: Save Query Results
#### What to Do
- Use Spark SQL to find locations with `total_cases` > 500000 and save the results (`location`, `total_cases`, `date`) to a CSV file.
#### Steps
- Write a SQL query to filter `total_cases > 500000`.
- Save to `output/high_cases_sql` using DataFrame API.
- Show 5 rows of the results.


In [73]:
result = spark.sql("""
SELECT location, total_cases, date
FROM covid_view
WHERE total_cases > 500000
""")

# Save result to CSV
result.write.option("header","true").mode("overwrite").csv("output/high_cases_sql")

# Show top 5 results
result.show(5)
result.count()

+---------+-----------+----------+
| location|total_cases|      date|
+---------+-----------+----------+
|Argentina|     524195|2020-09-06|
|Argentina|     524195|2020-09-07|
|Argentina|     524195|2020-09-08|
|Argentina|     524195|2020-09-09|
|Argentina|     524195|2020-09-10|
+---------+-----------+----------+
only showing top 5 rows



103726


---

## Finish Up
- Save as `exercises/04_exercise.ipynb`.
- Verify the file path for `covid-data.csv`.
- Ask your teacher if